In [0]:
%pip install -U ultralytics opencv-python-headless pandas mlflow requests

In [0]:
%restart_python

In [0]:
from pathlib import Path
import zipfile

ROOT = Path("/Volumes/workspace/default/people_sample")

print("Uploaded files:")
for file in ROOT.iterdir():
    print(file.name)

# Find the ZIP you uploaded
zip_files = list(ROOT.glob("*.zip"))

if not zip_files:
    raise FileNotFoundError("No ZIP found in people_sample")

ZIP_PATH = zip_files[0]
print("\nUsing:", ZIP_PATH.name)

# Inspect what's inside
with zipfile.ZipFile(ZIP_PATH) as z:
    names = [n for n in z.namelist() if not n.endswith("/")]

    print("\nTotal files:", len(names))
    print("\nFirst 30 files:")
    for name in names[:30]:
        print(name)

    videos = [
        n for n in names
        if n.lower().endswith((".mp4", ".mov", ".avi"))
    ]
    print("\nVideos found:", len(videos))

In [0]:

from pathlib import Path
import zipfile

ROOT = Path("/Volumes/workspace/default/people_sample")
ZIP_PATH = ROOT / "ETH-UCY.zip"

VIDEO_PATH = Path("/tmp/seq_eth.avi")

with zipfile.ZipFile(ZIP_PATH) as z:
    with z.open("videos/seq_eth.avi") as src:
        with open(VIDEO_PATH, "wb") as dst:
            import shutil
            shutil.copyfileobj(src, dst)

print("Video extracted:", VIDEO_PATH)
print("Size MB:", round(VIDEO_PATH.stat().st_size / 1e6, 2))


In [0]:
import json

summary_path = OUTPUT / f"{VIDEO_PATH.stem}_summary.json"

summary_path.write_text(
    json.dumps(summary, indent=2),
    encoding="utf-8"
)

mlflow.end_run()

with mlflow.start_run(
    run_name=f"people-tracking-{VIDEO_PATH.stem}"
) as run:
    mlflow.log_params({
        "detector": MODEL_NAME,
        "tracker": TRACKER,
        "confidence": CONFIDENCE,
        "iou_threshold": IOU_THRESHOLD,
        "prediction_model": "constant_velocity",
        "prediction_seconds": PREDICTION_SECONDS,
        "velocity_window_seconds": VELOCITY_WINDOW_SECONDS,
        "device": torch.cuda.get_device_name(0)
    })

    mlflow.log_metrics({
        "frames_processed": frame_index,
        "unique_people_tracked": unique_tracks,
        "track_observations": len(tracks),
        "trajectory_predictions": len(predictions)
    })

    mlflow.log_artifact(str(tracks_csv))
    mlflow.log_artifact(str(predictions_csv))
    mlflow.log_artifact(str(summary_path))

    print("MLflow run:", run.info.run_id)

print("Tracked video:", output_video)
print("Tracks:", tracks_csv)
print("Predictions:", predictions_csv)

In [0]:
%pip install ultralytics opencv-python-headless

In [0]:
import requests
import zipfile

ETH_UCY_URL = (
    "https://zenodo.org/records/19249591/"
    "files/ETH-UCY.zip?download=1"
)

ETH_UCY_ZIP = ROOT / "ETH-UCY.zip"
ETH_UCY_ROOT = ROOT / "ETH-UCY"

if not ETH_UCY_ZIP.exists():
    with requests.get(
        ETH_UCY_URL,
        stream=True,
        timeout=120
    ) as response:
        response.raise_for_status()

        with ETH_UCY_ZIP.open("wb") as target:
            for chunk in response.iter_content(
                chunk_size=8 * 1024 * 1024
            ):
                if chunk:
                    target.write(chunk)

if not ETH_UCY_ROOT.exists():
    ETH_UCY_ROOT.mkdir(
        parents=True,
        exist_ok=True
    )

    with zipfile.ZipFile(ETH_UCY_ZIP) as archive:
        archive.extractall(ETH_UCY_ROOT)

print("ZIP:", ETH_UCY_ZIP)
print("Dataset:", ETH_UCY_ROOT)
print(
    "Text files:",
    len(list(ETH_UCY_ROOT.rglob("*.txt")))
)

In [0]:
from ultralytics import YOLO
from pathlib import Path
import shutil

# Load pretrained YOLO
person_model = YOLO("yolo26n.pt")

# Convert to iPhone Core ML format
exported = Path(person_model.export(
    format="coreml",
    imgsz=640,
    nms=True
))

# Save permanently in your Databricks Volume
destination = Path(
    "/Volumes/workspace/default/people_sample/"
    "PersonDetection.mlpackage"
)

if destination.exists():
    shutil.rmtree(destination)

shutil.copytree(exported, destination)

print("Export successful!")
print("Saved to:", destination)


In [0]:
import shutil

shutil.make_archive(
    "/tmp/PersonDetection",
    "zip",
    root_dir=str(destination.parent),
    base_dir=destination.name
)

shutil.copy2(
    "/tmp/PersonDetection.zip",
    "/Volumes/workspace/default/people_sample/PersonDetection.zip"
)

print("Ready to download!")

In [0]:
from pathlib import Path
import zipfile
import shutil

ROOT = Path("/Volumes/workspace/default/people_sample")
VIDEO_PATH = Path("/tmp/seq_eth.avi")

with zipfile.ZipFile(ROOT / "ETH-UCY.zip") as z:
    with z.open("videos/seq_eth.avi") as source:
        with open(VIDEO_PATH, "wb") as output:
            shutil.copyfileobj(source, output)

print("Video ready:", VIDEO_PATH.exists())

In [0]:
%pip install ultralytics opencv-python-headless pandas

In [0]:

from pathlib import Path
from ultralytics import YOLO
import zipfile
import shutil
import cv2
import pandas as pd
import matplotlib.pyplot as plt

# 1. Locate the uploaded dataset
ROOT = Path("/Volumes/workspace/default/people_sample")
ZIP_PATH = ROOT / "ETH-UCY.zip"

# 2. Extract an actual pedestrian video
VIDEO_PATH = Path("/tmp/seq_eth_converted.mp4")

with zipfile.ZipFile(ZIP_PATH) as z:
    with z.open("videos/seq_eth.avi") as src:
        with open(VIDEO_PATH, "wb") as dst:
            shutil.copyfileobj(src, dst)

print("Video extracted:", VIDEO_PATH)

# 3. Read video information
cap = cv2.VideoCapture(str(VIDEO_PATH))
assert cap.isOpened(), "Could not open extracted video"

fps = cap.get(cv2.CAP_PROP_FPS)
fps = fps if fps > 0 else 25

width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
cap.release()

print("Video resolution:", width, "x", height)
print("FPS:", fps)

# 4. Load pretrained person detection model
model = YOLO("yolo26n.pt")

# 5. Prepare output
OUTPUT = ROOT / "tracking_output"
OUTPUT.mkdir(parents=True, exist_ok=True)

temp_video = "/tmp/people_tracked.mp4"

writer = cv2.VideoWriter(
    temp_video,
    cv2.VideoWriter_fourcc(*"mp4v"),
    fps,
    (width, height)
)
assert writer.isOpened(), "Video writer failed"

records = []

# 6. Run YOLO + BoT-SORT
results = model.track(
    source=str(VIDEO_PATH),
    tracker="botsort.yaml",
    classes=[0],          # Detect people only
    conf=0.25,
    stream=True,
    verbose=False
)

MAX_FRAMES = 200  # Set to None for full video

try:
    for frame_id, result in enumerate(results):
        if MAX_FRAMES is not None and frame_id >= MAX_FRAMES:
            break

        # Draw person boxes and tracking IDs
        annotated = result.plot()
        writer.write(annotated)

        boxes = result.boxes

        if boxes is None or boxes.id is None:
            continue

        ids = boxes.id.int().cpu().tolist()
        coords = boxes.xyxy.cpu().numpy()
        confidences = boxes.conf.cpu().numpy()

        for track_id, box, conf in zip(
            ids, coords, confidences
        ):
            x1, y1, x2, y2 = box

            records.append({
                "frame": frame_id,
                "time_s": frame_id / fps,
                "track_id": track_id,
                "x_px": float((x1 + x2) / 2),
                "y_px": float(y2),
                "confidence": float(conf)
            })

finally:
    writer.release()

# 7. Save annotated video and tracking data
video_out = OUTPUT / "people_tracked.mp4"
shutil.copyfile(temp_video, video_out)

columns = [
    "frame", "time_s", "track_id",
    "x_px", "y_px", "confidence"
]

df = pd.DataFrame(records, columns=columns)

csv_out = OUTPUT / "pedestrian_tracks.csv"
df.to_csv(csv_out, index=False)

print("\nTRACKING FINISHED!")
print("Tracked observations:", len(df))
print("Unique IDs:", df["track_id"].nunique())
print("Saved video:", video_out)
print("Saved CSV:", csv_out)

# 8. Display a preview frame
preview = cv2.VideoCapture(str(video_out))
success, frame = preview.read()
preview.release()

if success:
    plt.figure(figsize=(12, 7))
    plt.imshow(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
    plt.axis("off")
    plt.title("EchoPath — Pedestrian Tracking")
    plt.show()

display(df.head(10))


In [0]:
%pip install imageio-ffmpeg

In [0]:

from pathlib import Path
import zipfile
import shutil
import subprocess
import cv2
import imageio_ffmpeg

ROOT = Path("/Volumes/workspace/default/people_sample")

AVI_PATH = Path("/tmp/seq_eth.avi")
VIDEO_PATH = Path("/tmp/seq_eth_converted.mp4")

# Extract the real video, not the __MACOSX metadata
with zipfile.ZipFile(ROOT / "ETH-UCY.zip") as z:
    with z.open("videos/seq_eth.avi") as src:
        with AVI_PATH.open("wb") as dst:
            shutil.copyfileobj(src, dst)

print("AVI size:", round(AVI_PATH.stat().st_size / 1e6, 2), "MB")

# Convert to a commonly supported MP4 format
ffmpeg = imageio_ffmpeg.get_ffmpeg_exe()

result = subprocess.run(
    [
        ffmpeg,
        "-y",
        "-i", str(AVI_PATH),
        "-an",
        "-c:v", "mpeg4",
        "-q:v", "4",
        "-pix_fmt", "yuv420p",
        str(VIDEO_PATH)
    ],
    capture_output=True,
    text=True
)

if result.returncode != 0:
    print(result.stderr[-2500:])
    raise RuntimeError("FFmpeg conversion failed")

# Verify OpenCV can read the converted video
cap = cv2.VideoCapture(str(VIDEO_PATH))
success, frame = cap.read()
cap.release()

print("Converted video:", VIDEO_PATH)
print("OpenCV can read video:", success)

assert success, "Converted MP4 still cannot be decoded"


In [0]:

import base64
import io
import json
from pathlib import Path

import mlflow
import mlflow.pyfunc
import pandas as pd

from PIL import Image
from ultralytics import YOLO
from mlflow.models import infer_signature

mlflow.set_registry_uri("databricks-uc")

PEOPLE_VOLUME = Path(
    "/Volumes/workspace/default/people_sample"
)

existing_weights = (
    sorted(PEOPLE_VOLUME.rglob("yolo26n.pt"))
    if PEOPLE_VOLUME.exists()
    else []
)

if existing_weights:
    weights_path = existing_weights[0]
    print("Using existing weights:", weights_path)
else:
    print("Loading official pretrained YOLO26n")
    detector = YOLO("yolo26n.pt")

    weights_path = Path(
        "/tmp/lantern_person_yolo26n.pt"
    )

    detector.save(str(weights_path))

assert weights_path.exists()

class LanternPersonDetector(mlflow.pyfunc.PythonModel):

    def load_context(self, context):
        from ultralytics import YOLO

        self.detector = YOLO(
            context.artifacts["weights"]
        )

    def predict(self, context, model_input, params=None):
        import base64
        import io
        import json
        import pandas as pd

        from PIL import Image

        outputs = []

        for encoded_image in model_input["image_base64"]:

            image = Image.open(
                io.BytesIO(
                    base64.b64decode(encoded_image)
                )
            ).convert("RGB")

            prediction = self.detector.predict(
                source=image,
                imgsz=640,
                conf=0.25,
                iou=0.7,
                classes=[0],
                verbose=False,
                device="cpu"
            )[0]

            detections = []

            if prediction.boxes is not None:

                boxes = (
                    prediction.boxes.xyxyn
                    .detach()
                    .cpu()
                    .tolist()
                )

                scores = (
                    prediction.boxes.conf
                    .detach()
                    .cpu()
                    .tolist()
                )

                for box, score in zip(boxes, scores):
                    detections.append({
                        "bbox_xyxy_normalized": box,
                        "confidence": float(score),
                        "class_id": 0,
                        "class_name": "person"
                    })

            outputs.append(
                json.dumps(detections)
            )

        return pd.DataFrame({
            "person_detections_json": outputs
        })

signature = infer_signature(
    pd.DataFrame({
        "image_base64": [""]
    }),
    pd.DataFrame({
        "person_detections_json": ["[]"]
    })
)

artifacts = {
    "weights": str(weights_path)
}

coreml_candidates = (
    sorted(PEOPLE_VOLUME.rglob("PersonDetection.zip"))
    if PEOPLE_VOLUME.exists()
    else []
)

if coreml_candidates:
    artifacts["coreml_zip"] = str(
        coreml_candidates[0]
    )

with mlflow.start_run(
    run_name="lantern_person_model_registration"
):

    result = mlflow.pyfunc.log_model(
        name="lantern_person_detection",
        python_model=LanternPersonDetector(),
        artifacts=artifacts,
        signature=signature,
        pip_requirements=[
            "mlflow==3.17.2",
            "ultralytics==8.4.207",
            "torch",
            "numpy",
            "pandas",
            "pillow"
        ],
        registered_model_name=(
            "workspace.default.lantern_people"
        )
    )

print("PERSON MODEL REGISTERED")
print("Name: workspace.default.lantern_people")
print("Model URI:", result.model_uri)
print("Weights:", weights_path)
